In [5]:
from pathlib import Path
from collections import Counter

# -----------------------------------------------------
# DATASET PATH
# -----------------------------------------------------

DATASET_DIR = Path("../dataset")

# Class Names (same order as data.yaml)
CLASS_NAMES = {
    0: "Calcium Deficiency",
    1: "Healthy",
    2: "Magnesium Deficiency",
    3: "Nitrogen Deficiency",
    4: "Phosphorus Deficiency",
    5: "Potassium Deficiency"
}

SPLITS = ["train", "valid"]

class_counter = Counter()

print("=" * 70)
print("YOLO SEGMENTATION DATASET VALIDATION")
print("=" * 70)

for split in SPLITS:

    print(f"\n{'='*25} {split.upper()} SET {'='*25}")

    image_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"

    image_files = sorted(image_dir.glob("*.*"))
    label_files = sorted(label_dir.glob("*.txt"))

    print(f"Images : {len(image_files)}")
    print(f"Labels : {len(label_files)}")

    image_names = {img.stem for img in image_files}
    label_names = {lbl.stem for lbl in label_files}

    missing_labels = []
    extra_labels = []
    empty_labels = []
    invalid_labels = []

    # --------------------------------------------------
    # Missing Labels
    # --------------------------------------------------

    for img in image_files:

        label = label_dir / (img.stem + ".txt")

        if not label.exists():
            missing_labels.append(img.name)

    # --------------------------------------------------
    # Extra Labels
    # --------------------------------------------------

    for lbl in label_files:

        image_found = False

        for ext in [".jpg", ".jpeg", ".png", ".bmp"]:

            if (image_dir / (lbl.stem + ext)).exists():
                image_found = True
                break

        if not image_found:
            extra_labels.append(lbl.name)

    # --------------------------------------------------
    # Validate Every Label File
    # --------------------------------------------------

    for lbl in label_files:

        text = lbl.read_text().strip()

        if text == "":
            empty_labels.append(lbl.name)
            continue

        lines = text.splitlines()

        for line_no, line in enumerate(lines, start=1):

            values = line.strip().split()

            # Need class id + at least 3 points (1 class + 6 coords)
            if len(values) < 5:
                invalid_labels.append(
                    f"{lbl.name} (Line {line_no}) -> Skipped malformed polygon"
                )
                continue

            # -------------------------
            # Class ID
            # -------------------------

            try:
                class_id = int(values[0])

            except:
                invalid_labels.append(
                    f"{lbl.name} (Line {line_no}) -> Invalid class id"
                )
                continue

            if class_id not in CLASS_NAMES:
                invalid_labels.append(
                    f"{lbl.name} (Line {line_no}) -> Unknown class {class_id}"
                )
                continue

            class_counter[class_id] += 1

            # -------------------------
            # Polygon Coordinates
            # -------------------------

            try:
                coords = list(map(float, values[1:]))

            except:
                invalid_labels.append(
                    f"{lbl.name} (Line {line_no}) -> Non numeric coordinate"
                )
                continue

            # Even number of coordinates
            # Polygon must contain complete (x,y) pairs
            if len(coords) % 2 != 0:
                invalid_labels.append(
                f"{lbl.name} (Line {line_no}) -> Odd number of coordinates (Skipped)"
                )
                continue

            # Polygon should ideally have at least 3 points.
            # Tiny polygons with only 2 points are ignored.
            if len(coords) < 6:
                invalid_labels.append(
                f"{lbl.name} (Line {line_no}) -> Polygon has fewer than 3 points (Skipped)"
                )
                continue

            # Coordinates must be between 0 and 1
            bad_coord = False

            for c in coords:

                if c < 0 or c > 1:
                    bad_coord = True
                    break

            if bad_coord:

                invalid_labels.append(
                    f"{lbl.name} (Line {line_no}) -> Coordinate outside [0,1]"
                )

    # --------------------------------------------------
    # REPORT
    # --------------------------------------------------

    print(f"\nMissing Labels : {len(missing_labels)}")
    print(f"Extra Labels   : {len(extra_labels)}")
    print(f"Empty Labels   : {len(empty_labels)}")
    print(f"Invalid Labels : {len(invalid_labels)}")

    if missing_labels:

        print("\nMissing Label Files")

        for f in missing_labels:
            print("  ", f)

    if extra_labels:

        print("\nExtra Label Files")

        for f in extra_labels:
            print("  ", f)

    if empty_labels:

        print("\nEmpty Label Files")

        for f in empty_labels:
            print("  ", f)

    if invalid_labels:

        print("\nInvalid Label Details")

        for f in invalid_labels:
            print("  ", f)

# ---------------------------------------------------------
# CLASS DISTRIBUTION
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

total = sum(class_counter.values())

for cid in sorted(CLASS_NAMES.keys()):

    count = class_counter[cid]

    percentage = (count / total) * 100 if total else 0

    print(f"{CLASS_NAMES[cid]:25s} : {count:6d} ({percentage:5.2f}%)")

print("\nTotal Objects :", total)

print("\nValidation Completed Successfully.")

YOLO SEGMENTATION DATASET VALIDATION

========================= TRAIN SET =========================
Images : 2870
Labels : 2870

Missing Labels : 0
Extra Labels   : 0
Empty Labels   : 0
Invalid Labels : 3

Invalid Label Details
   IMG20230602082523_jpg.rf.62f9d30dbd507cd35cea1c28b6666554.txt (Line 15) -> Polygon has fewer than 3 points (Skipped)
   IMG20230602082523_jpg.rf.6df1eb31a88490578c3465ccca0f9739.txt (Line 15) -> Polygon has fewer than 3 points (Skipped)
   IMG20230602082523_jpg.rf.d0706f0cee0ae82cbf77a84e36bb7e40.txt (Line 15) -> Polygon has fewer than 3 points (Skipped)

========================= VALID SET =========================
Images : 107
Labels : 108

Missing Labels : 0
Extra Labels   : 1
Empty Labels   : 0
Invalid Labels : 0

Extra Label Files
   IMG20230602082256_jpg.rf.ac9e43de730bfa84db0ca8b0f759df81.txt

CLASS DISTRIBUTION
Calcium Deficiency        :   8133 (18.26%)
Healthy                   :   8083 (18.14%)
Magnesium Deficiency      :   5784 (12.98%)
Nitrogen D